# CSC786 A4 — Learned vs Hand-Weighted Trust Scoring Under Cross-Site Shift
Runs the full experiment pipeline on top of the `ztscore-telemetry` generator (v0.1.0). The generator-validation notebook is unchanged; this notebook adds the baseline, the learned models, the 4-condition × 5-seed matrix, the ablation, and the sensitivity check.

### 1. Setup
Upload `ztscore-telemetry.zip` (the version that includes `experiments/`).

In [ ]:
import os, sys, zipfile
try:
    from google.colab import files
    if not os.path.exists("ztscore-telemetry"):
        uploaded = files.upload()
        zipfile.ZipFile(next(iter(uploaded))).extractall(".")
    %pip install -q shap tabulate
except ImportError:
    pass
ROOT = "ztscore-telemetry" if os.path.exists("ztscore-telemetry") else "."
os.chdir(ROOT); sys.path.insert(0, ".")

### 2. Baseline: hand-weighted additive score (CSC773 Eqs. 1–3)
Weights, normalization anchors, floors, and the dynamic threshold are fixed a priori.

In [ ]:
from experiments import baseline as bl
import pandas as pd
print("Entity weights:", bl.ENTITY_WEIGHTS, "| floors:", bl.FLOORS, "| θ_mgmt =", bl.THETA_MGMT)
bl.effective_weights().sort_values(ascending=False).to_frame("effective weight")

### 3. Main matrix: A→A, A→B, B→B, B→A × 5 seeds
About 5 min per seed on Colab CPU. Set `SEEDS = [0]` for a quick check.

In [ ]:
from experiments.run_experiments import run
SEEDS = [0, 1, 2, 3, 4]
_ = run(SEEDS, "results/main")

### 4. Ablation: per-site re-baselining of channel and sync metrics

In [ ]:
_ = run(SEEDS, "results/ablation_rebaseline", rebaseline=True, do_shap=False, do_timing=False)

### 5. Generator sensitivity check (about 5 min)

In [ ]:
!python -m experiments.sensitivity --seeds 0 1 2 3 4 --out results/sensitivity

### 5b. Supplementary: 20-seed run for statistical power
Per-seed PR-AUC noise is about ±0.10–0.15 (only 5–9 compromised devices per test split), so the planned 5 seeds cannot resolve gap differences near 0.05. This repeats the main matrix and sensitivity check with seeds 0–19 (about 25 + 20 min).

In [ ]:
_ = run(list(range(20)), "results/main20")
!python -m experiments.sensitivity --seeds $(seq 0 19) --out results/sensitivity20

### 5c. Recall by kill-chain stage

In [ ]:
!python -m experiments.stage_recall --seeds 0 1 2 3 4 --out results/paper

### 6. Tables and figures for the report

In [ ]:
from experiments.aggregate import main as aggregate
from IPython.display import Image, Markdown, display
aggregate("results", "results/paper")                                    # planned 5-seed study
aggregate("results", "results/paper20", "main20", "sensitivity20")     # 20-seed supplementary

In [ ]:
for f in ["fig_gap", "fig_far_frr", "fig_shap", "fig_sensitivity"]:
    display(Markdown(f"**{f}**")); display(Image(f"results/paper/{f}.png", width=700))